#### Library imports

In [25]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import random
import scipy
import sys
import ipywidgets as widgets
import wandb


from typing import List, Tuple, Callable, Optional
from tqdm.notebook import tqdm
from ipywidgets import interact
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from torchvision.transforms import v2
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset
from torchvision.ops.feature_pyramid_network import LastLevelP6P7
from torchvision.models.detection import RetinaNet
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.resnet import ResNet34_Weights
from kaggle_secrets import UserSecretsClient
from torchmetrics.detection.mean_ap import MeanAveragePrecision

sys.path.append("/kaggle/input/datasets/alessandrotirelli/prw-utility-functions")
from utils import parse_annotations_file, draw_boxes

#### Parameters definition

In [26]:
do_train = False

batch_size = 4 # bs
num_epochs = 20 # ep
lr = 2e-5 # lr

lr_backbone_coeff = 0.5 # lr_backbone = lr * lr_backbone_coeff

num_workers = 4
log_interval = 10

seed = 42 # random seed

run_name = f"retina_resnet34_ep_{num_epochs}_bs_{batch_size}_lr_{lr}"
wandb_project = "ML4CV_assignment-PRW"

data_root = Path("/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild") 

#### GPU Initialization

In [27]:
device = "cpu"
if torch.cuda.is_available():
  print("All good, a Gpu is available.")
  device = torch.device("cuda:0")
else:
  print("Please set GPU via Edit -> Notebook Settings.")

All good, a Gpu is available.


#### Reproducibility and Deterministic Mode

In [28]:
def fix_random(seed: int) -> None:
    """Fix all the possible sources of randomness.

    Args:
        seed: the seed to use.
    """
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(seed=seed)

## Dataset and Dataloader

In [29]:
class PRWDataset(Dataset):
    def __init__(
        self,
        data_root: Path,
        split_str: str,
        transforms: Optional[Callable] = None
    ) -> None:
        """Init the PRW dataset.
    
        Args:
            data_root: the path to the folder containing all data.
            split_str: ["train" | "test"].
            transforms: the transformation to apply to the dataset.
        """
        
        ext_images="jpg"
        ext_annotations="jpg.mat"
        path_images=data_root/"frames"
        path_annotations=data_root/"annotations"
        
        if split_str == "train":  # train
            variable_name_split = "img_index_train"
            path_split=data_root/"frame_train.mat"
            
        elif split_str  == "test": # test
            variable_name_split = "img_index_test"
            path_split=data_root/"frame_test.mat"
        else: # fallback
            raise AssertionError(
                    f"{split_str} is not a valid value for `split_str` parameter. Please specify: 'train' or 'test'."
                )
    
        # read Matlab file
        mat = scipy.io.loadmat(path_split)
        sample_indexes_list = [str(elem[0][0]) for elem in mat[variable_name_split]] # in the string format: 'cX_sY_ZZZZZZ'
        
        # converting Python list to Python set to scale down access cost from O(n) to O(1) in list comprehension
        sample_indexes = set(sample_indexes_list)

        # DEBUG
        
        self.images = sorted([
            path for path in path_images.rglob(f"*.{ext_images}")
            if path.name.removesuffix(f".{ext_images}") in sample_indexes
        ])
        self.annotations = sorted([
            path for path in path_annotations.rglob(f"*.{ext_annotations}")
            if path.name.removesuffix(f".{ext_annotations}") in sample_indexes 
        ])
        self.transforms = transforms

        self.classes = ["__background__", "pedestrian"]
    
        if len(self.images) != len(self.annotations):
                raise AssertionError(
                    f"Images and labels differs in size: {len(self.images)} vs {len(self.annotations)}."
                )

    def __getitem__(self, idx):
        path_image = self.images[idx]
        path_annotations = self.annotations[idx]
    
        # load and convert images to Tensor
        image = Image.open(path_image).convert("RGB")
        if self.transforms is not None:
            image = self.transforms(image)
    
        # parse annotations and convert them to Tensors
        ids, boxes = parse_annotations_file(path_annotations)
        
        if len(boxes) == 0:
            boxes_xyxy = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
            ids = torch.zeros(0, dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)  # shape [N, 4] in [x, y, w, h]
            # Convert [x, y, w, h] → [x_min, y_min, x_max, y_max] (required by RetinaNet)
            boxes_xyxy = torch.stack([
                boxes[:, 0],               # x_min = x
                boxes[:, 1],               # y_min = y
                boxes[:, 0] + boxes[:, 2], # x_max = x + w
                boxes[:, 1] + boxes[:, 3], # y_max = y + h
            ], dim=1)  # your xywh→xyxy conversion
                
            # RetinaNet requires a "labels" key with class indices (1 = pedestrian, 0 = background)
            labels = torch.ones(len(ids), dtype=torch.int64)  # all boxes are "pedestrian"
            ids = torch.as_tensor(ids, dtype=torch.int64)
    
        target = {
            "boxes": boxes_xyxy,   # [N, 4] in xyxy format
            "labels": labels,      # [N] — required by RetinaNet
            "ids": ids             # [N] — your extra field, kept for reference
        }
    
        return image, target

    def __len__(self):
        return len(self.images)

In [30]:
# Create dataset
train_transform =  v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True)
])

data_train = PRWDataset(
    data_root=data_root,
    split_str="train",
    transforms=train_transform
)

classes = data_train.classes
num_classes = len(classes)

print(f"Number of training samples: {len(data_train)}")

Number of training samples: 5704


In [31]:
def collate_fn(batch):
    # https://discuss.pytorch.org/t/dataloader-gives-stack-expects-each-tensor-to-be-equal-size-due-to-different-image-has-different-objects-number/91941/4
    return tuple(zip(*batch))

print("DataLoaders creation stared...")
loader_train = DataLoader(
    data_train,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    num_workers=num_workers,
    collate_fn=collate_fn
)

DataLoaders creation stared...


## Training Data Visualization

In [32]:
if not do_train:
    # Visualize some images using ipywidgets interact() as a decorator
    @interact(
        sample_index = widgets.IntSlider(min=0, max=len(data_train)-1, step=1, value=0)
    )
    def plot_image_and_boxes(sample_index: int) -> None:
        """Plots an image from the training set with the respective bounding boxes.
    
        Args:
            sample_index: the index of the sampled image inside the training set.
        """
    
        image, target = data_train[sample_index]
        boxes = target["boxes"]
        pids = target["ids"]
        image = v2.ToPILImage()(image)
        
        cell_with_bb = draw_boxes(
            image,
            boxes=boxes,
            pids=pids,
            scores=[1.0] * len(boxes),
        )
        
        plt.imshow(cell_with_bb)
        plt.axis("off")
        plt.show()


interactive(children=(IntSlider(value=0, description='sample_index', max=5703), Output()), _dom_classes=('widg…

## RetinaNet Model Definition

In [33]:
backbone = resnet_fpn_backbone(
    backbone_name="resnet34",
    weights=ResNet34_Weights.IMAGENET1K_V1, # ResNet34 pretrained on ImageNet-1k as our backbone
    trainable_layers=5, # all ResNet layers are trainable (stem + C2 to C5)
    returned_layers=[2, 3, 4], # correspond to last three layers of ResNet, i.e. C3, C4 and C5, see image above
    extra_blocks=LastLevelP6P7(256, 256) # add levels P6 and P7 to get the RetinaNet backbone as defined in the paper
)

# RetinaNet paper (https://arxiv.org/pdf/1708.02002.pdf), section 4
anchor_scales = tuple(2 ** scale for scale in [0, 1/3, 2/3])
anchor_sizes = tuple(tuple(
    int(size * scale) for scale in anchor_scales) for size in [32, 64, 128, 256, 512]) # 5 sizes, one for each pyramid level (P3-P7)
anchor_ratios = ((1.5, 2.0, 3.0),) * len(anchor_sizes)  # Ratio = height / width (DIFFERENT FROM THE PAPER WHICH USED (0.5, 1.0, 2.0))

# https://github.com/pytorch/vision/blob/main/torchvision/models/detection/anchor_utils.py
anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=anchor_ratios)

# https://github.com/pytorch/vision/blob/main/torchvision/models/detection/retinanet.py#L323
retina_net = RetinaNet(
    backbone,
    num_classes,
    anchor_generator=anchor_generator,
    score_thresh=0.05,  # predictions with score <= this threshold are discarded
    nms_thresh=0.5,  # boxes with IoU > this threshold are discarded (used for non-maxima suppression)
    detections_per_img=100 # -1 was actually wrong  
).to(device)

 ## Training Loop 

In [34]:
def training_loop(
    model: nn.Module,
    num_epochs: int,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler,
    log_interval: int,
    device: torch.device,
    train_loader: DataLoader,
    val_loader: DataLoader,
) -> tuple:
    """Train a neural network with a fixed train/validation split.

    Args:
        model: the model to train.
        num_epochs: the number of epochs.
        optimizer: the optimizer used for training.
        scheduler: the learning rate scheduler.
        log_interval: every how many steps to log the metrics.
        device: the device to use to train the model.
        train_loader: data loader for the training split.
        val_loader: data loader for the validation split.

    Returns:
        Tuple containing the best validation map@0.5:0.95 and epoch.
    """
    global_step = 0
    best_map = -1.0
    best_epoch = 0
    best_model = None

    for epoch in tqdm(range(1, num_epochs + 1), desc="Training"):
        model.train()  # put model in training mode

        for idx_batch, (images, targets) in enumerate(train_loader):
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)

            loss_class = loss_dict["classification"]
            loss_box_regr = loss_dict["bbox_regression"]
            loss_sum = loss_class + loss_box_regr

            optimizer.zero_grad()
            loss_sum.backward()
            optimizer.step()
            scheduler.step()

            if log_interval > 0 and idx_batch % log_interval == 0:
                wandb.log({
                    "train/loss_sum": loss_sum.item(),
                    "train/loss_class": loss_class.item(),
                    "train/loss_box_regr": loss_box_regr.item(),
                    "train/lr": scheduler.get_last_lr()[0],
                    "train/epoch": epoch,
                }, step=global_step)

            global_step += 1

        # set the model to evaluation mode
        model.eval()

        # COCO-style AP metrics (IoU 0.50:0.95) to keep mAP in the original range
        map_metric = MeanAveragePrecision(
            iou_type="bbox",
            class_metrics=False,
        )

        with torch.no_grad():
            for images, targets in val_loader:
                images = [image.to(device) for image in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                predictions = model(images)
                preds = [
                    {
                        "boxes": pred["boxes"].cpu(),
                        "scores": pred["scores"].cpu(),
                        "labels": pred["labels"].cpu(),
                    }
                    for pred in predictions
                ]

                targs = [
                    {
                        "boxes": targ["boxes"].cpu(),
                        "labels": targ["labels"].cpu(),
                    }
                    for targ in targets
                ]

                map_metric.update(preds, targs)

        map_metrics = map_metric.compute()

        current_map = map_metrics["map"].item()

        print(f"Epoch {epoch} done. Logging to WandB...")
        wandb.log({
            "val/map@0.5:0.95": current_map,
            "val/map_small": map_metrics["map_small"],
            "val/map_medium": map_metrics["map_medium"],
            "val/map_large": map_metrics["map_large"],
            "val/epoch": epoch,
        }, step=global_step)

        if current_map > best_map:
            print("New best model found (by map@0.5:0.95). Saving best model...")
            best_map = current_map
            best_epoch = epoch
            best_model = copy.deepcopy(model)

    return best_map, best_epoch, best_model

In [35]:
def train(
    model: nn.Module,
    num_epochs: int,
    max_lr: float,
    log_interval: int,
    device: torch.device,
    run_name: str,
    wandb_project: str,
    train_loader: DataLoader,
    val_loader: DataLoader = None,
) -> None:
    """Executes the training loop with one fixed train/validation split.

    Args:
        model: the model to train.
        num_epochs: the number of epochs.
        max_lr: the maximum learning rate.
        log_interval: every how many steps to log the metrics.
        device: the device to use to train the model.
        run_name: the wandb run name.
        wandb_project: the wandb project name.
        train_loader: the data loader containing the full training data.
        val_loader: optional validation loader. If None, it is created from train_loader with a fixed split.
    """

    if val_loader is None:
        base_loader = train_loader
        dataset = base_loader.dataset

        if len(dataset) < 2:
            raise ValueError("At least 2 samples are required to create train/val split.")

        # train/val split
        val_ratio = 0.2 #80/20 split
        num_samples = len(dataset)
        num_val = max(1, int(num_samples * val_ratio))
        num_train = num_samples - num_val
    
        split_gen = torch.Generator().manual_seed(seed) # fixed seed for reproducibility
        shuffled_indices = torch.randperm(num_samples, generator=split_gen).tolist()
        train_indices = shuffled_indices[:num_train]
        val_indices = shuffled_indices[num_train:]

        train_subset = torch.utils.data.Subset(dataset, train_indices)
        val_subset = torch.utils.data.Subset(dataset, val_indices)

        train_loader = DataLoader(
            train_subset,
            batch_size=base_loader.batch_size,
            shuffle=True,  # reshuffle every epoch
            pin_memory=True,
            num_workers=base_loader.num_workers,
            collate_fn=base_loader.collate_fn,
        )

        val_loader = DataLoader(
            val_subset,
            batch_size=base_loader.batch_size,
            shuffle=False,
            pin_memory=True,
            num_workers=base_loader.num_workers,
            collate_fn=base_loader.collate_fn,
        )

        print(f"Fixed split created: train={len(train_subset)}, val={len(val_subset)}")

    print("Model built. Starting WandB login...")
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("wandb_api_key")
    wandb.login(key=wandb_key)
    print("WandB login OK. Initializing run...")
    wandb.init(
        project=wandb_project,
        name=run_name,
        settings=wandb.Settings(_disable_stats=True),
    )
    print("WandB init OK. Starting training loop...")

    # Optimization (backbone parameters are learned with 10% learning rate)
    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if "body" in name or "fpn" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = optim.Adam([
        {"params": backbone_params, "lr": max_lr * lr_backbone_coeff},
        {"params": head_params, "lr": max_lr},
    ])

    total_steps = num_epochs * len(train_loader)
    scheduler = lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        total_steps=total_steps,
        div_factor=10,
        final_div_factor=100,
    )

    best_map, best_epoch, best_model = training_loop(
        model=model,
        num_epochs=num_epochs,
        optimizer=optimizer,
        scheduler=scheduler,
        log_interval=log_interval,
        device=device,
        train_loader=train_loader,
        val_loader=val_loader,
    )
    wandb.finish()

    path_ckpts = Path("/kaggle/working/ckpts")
    path_ckpts.mkdir(exist_ok=True)
    torch.save(best_model.state_dict(), path_ckpts / f"{run_name}.pt")
    print(f"Saved best model at epoch {best_epoch} with map@0.5:0.95: {best_map:.4f}")

## Train Step

In [36]:
if do_train:
    train(
        retina_net,
        num_epochs,
        lr,
        log_interval,
        device,
        run_name,
        wandb_project,
        loader_train,
    )
else:
    print("Training skipped.")

Training skipped.
